# 3C Non-Gaussian Forward Model Sanity Check

This notebook checks:
1. `S(0,0)=1`
2. `W` sum = 1
3. No-exchange gives diagonal-only pathways
4. Increasing permeability / mixing time increases DEI
5. `K_E, K_T, K_S` vs `b`, and restricted kernels are not simple exponentials

Also demonstrates new utilities: `compute_dei_from_weight_matrix`, `add_rician_noise`, `sample_dataset`.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
repo_root = None
for c in candidate_roots:
    if (c / 'dexsy_core').exists():
        repo_root = c
        break
if repo_root is None:
    raise RuntimeError('Could not locate repo root containing dexsy_core.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dexsy_core import ForwardModel3CNonGaussian

np.set_printoptions(precision=4, suppress=True)
print('Repo root:', repo_root)


In [ ]:
fm = ForwardModel3CNonGaussian(n_b=16, n_restrict_terms=500)

phi = np.array([0.45, 0.35, 0.20], dtype=np.float64)  # E,T,S
mixing_time = 0.08
D_E = 1.7e-9
D_I = 0.7e-9
l_T = 1.0e-6
R_S = 4.0e-6

q_default = fm.build_generator(k_et=3.0, k_te=2.0, k_es=1.5, k_se=1.2, k_ts=0.0, k_st=0.0)

signal, details = fm.compute_signal(
    phi=phi,
    q=q_default,
    mixing_time=mixing_time,
    extracellular_diffusivity=D_E,
    intracellular_diffusivity=D_I,
    axon_restricted_length=l_T,
    sphere_radius=R_S,
)

W = details['weight_matrix']
dei = fm.compute_dei_from_weight_matrix(W)
print('Signal shape:', signal.shape)
print('Initial DEI:', dei)


In [ ]:
# 1) Check S(0,0)=1
ok1 = np.isclose(signal[0, 0], 1.0, atol=1e-10)
print('1) S(0,0)=1 ->', 'PASS' if ok1 else 'FAIL', '| value =', float(signal[0, 0]))

# 2) Check W sum = 1
ok2 = np.isclose(W.sum(), 1.0, atol=1e-10)
print('2) W sum = 1 ->', 'PASS' if ok2 else 'FAIL', '| value =', float(W.sum()))


In [ ]:
# 3) No exchange => only diagonal pathways (EE, TT, SS)
q_zero = np.zeros((3, 3), dtype=np.float64)
_sig0, det0 = fm.compute_signal(
    phi=phi,
    q=q_zero,
    mixing_time=mixing_time,
    extracellular_diffusivity=D_E,
    intracellular_diffusivity=D_I,
    axon_restricted_length=l_T,
    sphere_radius=R_S,
)
W0 = det0['weight_matrix']
offdiag_sum = float(W0.sum() - np.trace(W0))
ok3 = offdiag_sum < 1e-12
print('3) No exchange -> diagonal only:', 'PASS' if ok3 else 'FAIL', '| offdiag sum =', offdiag_sum)
print('W0 =
', W0)


In [ ]:
# 4a) Increase permeability => DEI increases (fixed mixing time)
q_low, rates_low = fm.build_generator_from_permeability(
    phi=phi,
    sphere_diameter=2*R_S,
    sphere_permeability=2e-6,
    axon_surface_to_volume=1.5e6,
    axon_permeability=2e-7,
)
q_high, rates_high = fm.build_generator_from_permeability(
    phi=phi,
    sphere_diameter=2*R_S,
    sphere_permeability=2e-5,
    axon_surface_to_volume=1.5e6,
    axon_permeability=2e-6,
)

_, d_low = fm.compute_signal(phi=phi, q=q_low, mixing_time=0.08, extracellular_diffusivity=D_E, intracellular_diffusivity=D_I, axon_restricted_length=l_T, sphere_radius=R_S)
_, d_high = fm.compute_signal(phi=phi, q=q_high, mixing_time=0.08, extracellular_diffusivity=D_E, intracellular_diffusivity=D_I, axon_restricted_length=l_T, sphere_radius=R_S)
dei_low = fm.compute_dei_from_weight_matrix(d_low['weight_matrix'])
dei_high = fm.compute_dei_from_weight_matrix(d_high['weight_matrix'])
ok4a = dei_high > dei_low
print('4a) Permeability up -> DEI up:', 'PASS' if ok4a else 'FAIL', '| low=', dei_low, 'high=', dei_high)

# 4b) Increase mixing time => DEI increases (fixed permeability)
_, d_tm_low = fm.compute_signal(phi=phi, q=q_high, mixing_time=0.01, extracellular_diffusivity=D_E, intracellular_diffusivity=D_I, axon_restricted_length=l_T, sphere_radius=R_S)
_, d_tm_high = fm.compute_signal(phi=phi, q=q_high, mixing_time=0.20, extracellular_diffusivity=D_E, intracellular_diffusivity=D_I, axon_restricted_length=l_T, sphere_radius=R_S)
dei_tm_low = fm.compute_dei_from_weight_matrix(d_tm_low['weight_matrix'])
dei_tm_high = fm.compute_dei_from_weight_matrix(d_tm_high['weight_matrix'])
ok4b = dei_tm_high > dei_tm_low
print('4b) Mixing time up -> DEI up:', 'PASS' if ok4b else 'FAIL', '| low=', dei_tm_low, 'high=', dei_tm_high)


In [ ]:
# 5) Plot K_E, K_T, K_S vs b and check non-exponential behavior for restricted kernels
kern = fm.compartment_kernels(
    g=fm.G1,
    extracellular_diffusivity=D_E,
    intracellular_diffusivity=D_I,
    axon_restricted_length=l_T,
    sphere_radius=R_S,
)
b = fm.b1
KE, KT, KS = kern['E'], kern['T'], kern['S']

def monoexp_fit_residual(b, k):
    mask = (b > 0) & (k > 0)
    x = b[mask]
    y = np.log(k[mask])
    A = np.vstack([x, np.ones_like(x)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    y_hat = slope * x + intercept
    rmse = np.sqrt(np.mean((y - y_hat) ** 2))
    return float(rmse), float(slope), float(intercept)

rmse_e, *_ = monoexp_fit_residual(b, KE)
rmse_t, *_ = monoexp_fit_residual(b, KT)
rmse_s, *_ = monoexp_fit_residual(b, KS)

print('log-domain monoexp RMSE:')
print('  E (Gaussian)   :', rmse_e)
print('  T (restricted) :', rmse_t)
print('  S (restricted) :', rmse_s)
print('Restricted non-exponential check (expect T/S > E):', 'PASS' if (rmse_t > rmse_e and rmse_s > rmse_e) else 'CHECK')

plt.figure(figsize=(7.5, 5.2))
plt.plot(b, KE, label='K_E (Gaussian)', lw=2)
plt.plot(b, KT, label='K_T (restricted)', lw=2)
plt.plot(b, KS, label='K_S (restricted)', lw=2)
plt.xlabel('b (s/m^2)')
plt.ylabel('Kernel amplitude')
plt.title('Compartment Kernels vs b')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# New utility demos: Rician noise + dataset sampling
signal_clean_un, _ = fm.compute_signal(
    phi=phi, q=q_default, mixing_time=0.08,
    extracellular_diffusivity=D_E, intracellular_diffusivity=D_I,
    axon_restricted_length=l_T, sphere_radius=R_S,
    normalize=False
)
signal_noisy = fm.add_rician_noise(signal_clean_un, noise_sigma=0.01, normalize=True)
print('Rician noisy S(0,0)=', float(signal_noisy[0,0]))

signals, params, clean = fm.sample_dataset(
    n_samples=8,
    noise_sigma_range=(0.005, 0.015),
    seed=42,
    return_clean_signals=True,
)
print('Dataset shapes -> noisy:', signals.shape, ', clean:', clean.shape, ', n_params:', len(params))
print('First sample DEI (weight-matrix):', params[0]['dei_weight_matrix'])
